In [1]:
from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Config

In [ ]:
BASE_DIR = Path(r"./experiments_results/experiments_one_stage")

TARGET_FILE_STEMS = ("test_metrics_model_space",)

EXCLUDED_DIR_PREFIXES = ("experiments",)

OUTPUT_DIR = BASE_DIR / "model_space_metrics_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = [
    "loss",
    "macro_f1",
    "balanced_acc",
    "overall_acc",
]

COUNT_COL = "count"

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: experiments_results\experiments_one_stage
OUTPUT_DIR: experiments_results\experiments_one_stage\model_space_metrics_analysis


Method name and seed parser

In [ ]:
def clean_method_name(name: str) -> str:
    name = re.sub(r"[\s\-]+", "_", str(name))
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def extract_seed_and_method(folder_name: str):
    name = str(folder_name).strip()

    patterns = [
        r"(?i)(?:^|[_\-\s])seed[_\-\s]*(\d+)(?=$|[_\-\s])",
        r"(?i)(?:^|[_\-\s])(\d+)[_\-\s]*seed(?=$|[_\-\s])",
    ]

    for pat in patterns:
        m = re.search(pat, name)
        if m:
            seed = int(m.group(1))
            method = name[:m.start()] + name[m.end():]
            return clean_method_name(method), seed

    return clean_method_name(name), np.nan


for example in [
    "two_stage_ast_mfcc_spec_seed_67",
    "two_stage_ast_mfcc_spec_seed_267",
    "ast_checkpoint_speed_67_seed",
    "ast_checkpoint_none_mfcc_167_seed",
    "ast_checkpoint_base_67_seed",
    "ast_checkpoint_back_267_seed",
    "head_4_seed_67",
    "head_8_seed_267",
]:
    print(example, "->", extract_seed_and_method(example))

two_stage_ast_mfcc_spec_seed_67 -> ('two_stage_ast_mfcc_spec', 67)
two_stage_ast_mfcc_spec_seed_267 -> ('two_stage_ast_mfcc_spec', 267)
ast_checkpoint_speed_67_seed -> ('ast_checkpoint_speed', 67)
ast_checkpoint_none_mfcc_167_seed -> ('ast_checkpoint_none_mfcc', 167)
ast_checkpoint_base_67_seed -> ('ast_checkpoint_base', 67)
ast_checkpoint_back_267_seed -> ('ast_checkpoint_back', 267)
head_4_seed_67 -> ('head_4', 67)
head_8_seed_267 -> ('head_8', 267)


In [ ]:
def path_has_excluded_dir(path: Path, base_dir: Path, excluded_prefixes=EXCLUDED_DIR_PREFIXES) -> bool:
    try:
        rel_parts = path.relative_to(base_dir).parts
    except ValueError:
        rel_parts = path.parts

    for part in rel_parts[:-1]:
        part_lower = part.lower()
        if any(part_lower.startswith(prefix.lower()) for prefix in excluded_prefixes):
            return True

    return False


def is_target_metrics_file(path: Path) -> bool:
    name = path.name
    stem = path.stem

    return name in TARGET_FILE_STEMS or stem in TARGET_FILE_STEMS


def find_metric_files(base_dir: Path) -> pd.DataFrame:
    all_candidate_paths = [
        p for p in base_dir.rglob("*")
        if p.is_file() and is_target_metrics_file(p)
    ]

    rows = []
    for path in sorted(all_candidate_paths):
        excluded = path_has_excluded_dir(path, base_dir)

        rows.append({
            "path": path,
            "path_str": str(path),
            "excluded": excluded,
            "reason": "excluded_dir_prefix" if excluded else "",
        })

    return pd.DataFrame(rows)


found_files = find_metric_files(BASE_DIR)

print("Znalezione pliki łącznie:", len(found_files))
if not found_files.empty:
    print("Uwzględnione:", int((~found_files["excluded"]).sum()))
    print("Pominięte:", int(found_files["excluded"].sum()))

display(found_files)

Znalezione pliki łącznie: 3
Uwzględnione: 3
Pominięte: 0


,path,path_str,excluded,reason
0,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,False,
1,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,False,
2,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,False,


In [ ]:
def find_nearest_folder_with_seed(metrics_file_path: Path, base_dir: Path):
    p = metrics_file_path.parent

    while True:
        method, seed = extract_seed_and_method(p.name)
        if not pd.isna(seed):
            return p, method, seed

        if p == base_dir or p.parent == p:
            break

        p = p.parent

    method, seed = extract_seed_and_method(metrics_file_path.parent.name)
    return metrics_file_path.parent, method, seed


def preview_run_folder_detection(files_df: pd.DataFrame):
    rows = []

    for path in files_df.loc[~files_df["excluded"], "path"]:
        run_folder, method, seed = find_nearest_folder_with_seed(path, BASE_DIR)
        rows.append({
            "file": str(path),
            "run_folder": str(run_folder),
            "run_folder_name": run_folder.name,
            "method": method,
            "seed": seed,
        })

    return pd.DataFrame(rows)


run_detection_preview = preview_run_folder_detection(found_files)
display(run_detection_preview.sort_values(["method", "seed", "file"]))

,file,run_folder,run_folder_name,method,seed
2,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_67,single_stage_ast_mfcc_none,67
0,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_167,single_stage_ast_mfcc_none,167
1,experiments_results\experiments_one_stage\sing...,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_267,single_stage_ast_mfcc_none,267


Reading all metrics

In [ ]:
def read_metrics_file(path: Path) -> pd.DataFrame:
    
    df = pd.read_csv(path)

    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

    required = set(METRICS + [COUNT_COL])
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Plik {path} nie ma wymaganych kolumn: {missing}. Kolumny: {list(df.columns)}")

    return df


def load_all_metrics(files_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    included_paths = files_df.loc[~files_df["excluded"], "path"].tolist()

    if len(included_paths) == 0:
        raise FileNotFoundError(
            f"Nie znaleziono żadnych plików {TARGET_FILE_STEMS} po filtrach w {BASE_DIR}"
        )

    for path in included_paths:
        run_folder, method, seed = find_nearest_folder_with_seed(path, BASE_DIR)

        df = read_metrics_file(path)

        if len(df) > 1:
            warnings.warn(f"Plik {path} ma {len(df)} wierszy. Biorę ostatni wiersz.")
            df = df.tail(1).copy()

        row = df.iloc[0].to_dict()

        row.update({
            "method": method,
            "seed": seed,
            "run_folder": str(run_folder),
            "run_folder_name": run_folder.name,
            "metrics_path": str(path),
        })

        rows.append(row)

    all_metrics = pd.DataFrame(rows)

    for col in METRICS + [COUNT_COL, "seed"]:
        if col in all_metrics.columns:
            all_metrics[col] = pd.to_numeric(all_metrics[col], errors="coerce")

    all_metrics = all_metrics.sort_values(["method", "seed", "run_folder_name"]).reset_index(drop=True)

    return all_metrics


all_metrics = load_all_metrics(found_files)

print("Liczba runów:", len(all_metrics))
print("Liczba metod:", all_metrics["method"].nunique())
display(all_metrics)

Liczba runów: 3
Liczba metod: 1


,loss,macro_f1,balanced_acc,overall_acc,count,method,seed,run_folder,run_folder_name,metrics_path
0,0.210894,0.871654,0.950488,0.950762,2823.0,single_stage_ast_mfcc_none,67,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_67,experiments_results\experiments_one_stage\sing...
1,0.231356,0.868833,0.947541,0.947574,2823.0,single_stage_ast_mfcc_none,167,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_167,experiments_results\experiments_one_stage\sing...
2,0.263692,0.865080,0.943538,0.943677,2823.0,single_stage_ast_mfcc_none,267,experiments_results\experiments_one_stage\sing...,single_stage_ast_mfcc_none_seed_267,experiments_results\experiments_one_stage\sing...


Duplicate check

In [10]:
dupes = (
    all_metrics
    .groupby(["method", "seed"], dropna=False)
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

if len(dupes) > 0:
    print("UWAGA: wykryto duplikaty method + seed.")
    print("Notebook uśredni duplikaty przed agregacją metod.")
    display(dupes)

    display(
        all_metrics.merge(dupes[["method", "seed"]], on=["method", "seed"], how="inner")
        .sort_values(["method", "seed", "run_folder_name"])
    )
else:
    print("OK: brak duplikatów method + seed.")

OK: brak duplikatów method + seed.


Method aggregation

In [ ]:
per_seed = (
    all_metrics
    .groupby(["method", "seed"], dropna=False)
    .agg(
        loss=("loss", "mean"),
        macro_f1=("macro_f1", "mean"),
        balanced_acc=("balanced_acc", "mean"),
        overall_acc=("overall_acc", "mean"),
        count=("count", "mean"),
        n_files=("metrics_path", "count"),
        run_folders=("run_folder_name", lambda x: " | ".join(map(str, x))),
    )
    .reset_index()
)

agg_parts = []

for metric in METRICS:
    part = (
        per_seed
        .groupby("method", dropna=False)[metric]
        .agg(["mean", "std", "min", "max", "median", "count"])
        .reset_index()
    )

    part = part.rename(columns={
        "mean": f"{metric}_mean",
        "std": f"{metric}_std",
        "min": f"{metric}_min",
        "max": f"{metric}_max",
        "median": f"{metric}_median",
        "count": f"{metric}_n",
    })

    agg_parts.append(part)

summary = agg_parts[0]
for part in agg_parts[1:]:
    summary = summary.merge(part, on="method", how="outer")

summary["n_seeds"] = per_seed.groupby("method")["seed"].nunique(dropna=True).reindex(summary["method"]).to_numpy()
summary["n_runs"] = per_seed.groupby("method")["n_files"].sum().reindex(summary["method"]).to_numpy()

summary = summary.sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)

display(summary)

,method,loss_mean,loss_std,loss_min,loss_max,loss_median,loss_n,macro_f1_mean,macro_f1_std,macro_f1_min,...,balanced_acc_median,balanced_acc_n,overall_acc_mean,overall_acc_std,overall_acc_min,overall_acc_max,overall_acc_median,overall_acc_n,n_seeds,n_runs
0,single_stage_ast_mfcc_none,0.235314,0.026621,0.210894,0.263692,0.231356,3,0.868522,0.003298,0.86508,...,0.947541,3,0.947337,0.003548,0.943677,0.950762,0.947574,3,3,3


In [12]:
def mean_std_str(mean, std, digits=4):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


pretty_summary = pd.DataFrame({
    "method": summary["method"],
    "n_seeds": summary["n_seeds"],
    "loss": [
        mean_std_str(m, s) for m, s in zip(summary["loss_mean"], summary["loss_std"])
    ],
    "macro_f1": [
        mean_std_str(m, s) for m, s in zip(summary["macro_f1_mean"], summary["macro_f1_std"])
    ],
    "balanced_acc": [
        mean_std_str(m, s) for m, s in zip(summary["balanced_acc_mean"], summary["balanced_acc_std"])
    ],
    "overall_acc": [
        mean_std_str(m, s) for m, s in zip(summary["overall_acc_mean"], summary["overall_acc_std"])
    ],
})

display(pretty_summary)

,method,n_seeds,loss,macro_f1,balanced_acc,overall_acc
0,single_stage_ast_mfcc_none,3,0.2353 ± 0.0266,0.8685 ± 0.0033,0.9472 ± 0.0035,0.9473 ± 0.0035


CSV File

In [ ]:
all_metrics.to_csv(OUTPUT_DIR / "all_model_space_metrics_raw.csv", index=False)
per_seed.to_csv(OUTPUT_DIR / "model_space_metrics_per_seed.csv", index=False)
summary.to_csv(OUTPUT_DIR / "model_space_metrics_summary.csv", index=False)
pretty_summary.to_csv(OUTPUT_DIR / "model_space_metrics_summary_pretty.csv", index=False)

xlsx_path = OUTPUT_DIR / "model_space_metrics_analysis.xlsx"

with pd.ExcelWriter(xlsx_path) as writer:
    all_metrics.to_excel(writer, sheet_name="raw", index=False)
    per_seed.to_excel(writer, sheet_name="per_seed", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    pretty_summary.to_excel(writer, sheet_name="pretty", index=False)

print("Zapisano do:", OUTPUT_DIR)
print("Excel:", xlsx_path)